# FINPLE canonical CSV monthly metrics build

This notebook is a thin Colab runner for the repository pipeline. It builds a full-schema candidate artifact only; it never replaces `src/data/tickers/finple_app_candidates_v2.csv`. The editable universe must come from the current bootstrap/update tools and contain separate `providerSymbol` and `marketDataProviderSymbol` fields. To reuse cache after a Colab restart, mount Drive in a separate operator cell (`from google.colab import drive; drive.mount('/content/drive')`) before running the pipeline. If Drive is not mounted, change `CACHE_DIR` and `UNIVERSE_PATH` to `/content/...` paths.

In [ ]:
# Edit only these run inputs.
AS_OF_DATE = "2026-07-29"  # Required; explicit completed date only.
SOURCE_CANONICAL_PATH = "src/data/tickers/finple_app_candidates_v2.csv"
UNIVERSE_PATH = "/content/drive/MyDrive/FINPLE/editable-universe.csv"
OUTPUT_CANDIDATE_PATH = "outputs/finple_app_candidates_v2.candidate.csv"
CACHE_DIR = "/content/drive/MyDrive/FINPLE/canonical_csv_cache"
CHUNK_SIZE = 100
RESUME = True
RETRY_COUNT = 3
RETRY_BACKOFF_SECONDS = 5
ROLLING_CAGR_WINDOW_YEARS = (10, 7, 5, 3, 1)
MIN_ROLLING_WINDOWS = 6

In [ ]:
# Colab dependency for the optional live provider.
%pip install -q yfinance

In [ ]:
from pathlib import Path

from tools.canonical_csv.build import build_canonical_candidate
from tools.canonical_csv.cache import PersistentCachedMarketDataProvider
from tools.canonical_csv.config import PipelineConfig
from tools.canonical_csv.market_data import YFinanceMarketDataProvider

In [ ]:
config = PipelineConfig.from_strings(
    source_canonical_path=SOURCE_CANONICAL_PATH,
    universe_path=UNIVERSE_PATH,
    output_candidate_path=OUTPUT_CANDIDATE_PATH,
    as_of_date=AS_OF_DATE,
    cache_dir=Path(CACHE_DIR),
    chunk_size=CHUNK_SIZE,
    resume=RESUME,
    retry_count=RETRY_COUNT,
    retry_backoff_seconds=RETRY_BACKOFF_SECONDS,
    rolling_cagr_window_years=ROLLING_CAGR_WINDOW_YEARS,
    min_rolling_windows=MIN_ROLLING_WINDOWS,
)
provider = PersistentCachedMarketDataProvider(
    YFinanceMarketDataProvider(),
    config.cache_dir,
    retry_count=config.retry_count,
    retry_backoff_seconds=config.retry_backoff_seconds,
)
result = build_canonical_candidate(config, provider)
result.summary

In [ ]:
from google.colab import files

for artifact in (
    result.candidate_path,
    result.validation_report_path,
    result.failed_assets_path,
    result.run_summary_path,
):
    files.download(str(artifact))